## NEOs close-approaches — construcción del dataset y etiquetado PHA

Este proyecto estudia los Objetos Cercanos a la Tierra (NEOs) a partir de su
registro histórico de **aproximaciones cercanas** (1900–presente). Combina un
análisis exploratorio no supervisado (PCA, K-Means, t-SNE) con la pregunta
central del trabajo: **¿puede inferirse el carácter potencialmente peligroso
(PHA) de un NEO solo desde la cinemática de sus aproximaciones observadas, y
cómo distorsiona esa inferencia la función de selección observacional del
catálogo?**

Este notebook construye el dataset (descarga desde las APIs de JPL, limpieza y
etiquetado). El análisis vive en `notebooks/ProyectoNeoRework_ml.ipynb`.

# Data
Los datos fueron extraídos mediante el consumo de la API proporcionada por el sistema de monitoreo de aproximaciones cercanas a la Tierra (Close-Approach Data) del Center for Near Earth Object Studies, perteneciente a la NASA. La fuente oficial se encuentra disponible en: https://cneos.jpl.nasa.gov/ca/

## Librerias data

In [1]:
import pandas as pd
import os
import time
from datetime import datetime
import re
import requests
import math

## Carga y preparacion de datos

In [2]:
# Carpeta de destino de los datos: esta misma carpeta (data/).
# El notebook debe ejecutarse con data/ como directorio de trabajo.
carpeta_destino = "."
os.makedirs(carpeta_destino, exist_ok=True)
ruta_csv = os.path.join(carpeta_destino, "close_approaches.csv")

In [3]:
# Funciones

def archivo_es_reciente(ruta, dias_maximos=30):
    """Devuelve True si el archivo fue modificado hace menos de dias_maximos días."""
    if not os.path.exists(ruta):
        return False
    edad_segundos = time.time() - os.path.getmtime(ruta)
    return edad_segundos < dias_maximos * 86400


def limpiar_fecha(fecha_str):
    """Elimina sufijos de incertidumbre como ±00:01 en las fechas de la API."""
    if isinstance(fecha_str, str):
        return fecha_str.split("±")[0].strip()
    return fecha_str


class ErrorAPI(RuntimeError):
    """La API de JPL respondio algo que no se puede usar como dataset."""


def payload_json(resp, contexto):
    """Parsea la respuesta de una API de JPL y comprueba que sea utilizable.

    Dos fallos no dan codigo HTTP de error y por tanto `raise_for_status` no los
    ve: (1) la API responde 200 con {"message": ...} cuando la consulta es
    invalida, y (2) la respuesta llega cortada, con lo que el JSON no parsea.
    Sin este control, el primero acaba en un KeyError sobre "data" y el segundo
    en un ValueError de json, ninguno de los dos mencionando la peticion.
    """
    try:
        js = resp.json()
    except ValueError as exc:
        raise ErrorAPI(f"{contexto}: la respuesta no es JSON válido ({exc}); "
                       f"empieza por {resp.text[:200]!r}") from exc
    if "data" not in js and "message" in js:
        raise ErrorAPI(f"{contexto}: la API devolvió un error: {js['message']}")
    return js


def descargar_cad(dist_max, anio_ini=1900, paso=10, intentos=3, timeout=300):
    """Descarga aproximaciones cercanas de la CAD API por tramos de `paso` anios.

    La consulta completa (~340k eventos) en una sola peticion falla de forma
    intermitente por corte de la respuesta; trocearla la hace reproducible y,
    de paso, unas 3 veces mas rapida.
    """
    anio_fin = int(datetime.utcnow().strftime("%Y")) + 1
    tramos = [(a, min(a + paso, anio_fin)) for a in range(anio_ini, anio_fin, paso)]
    filas, campos, vacios = [], None, []

    for a, b in tramos:
        for intento in range(intentos):
            try:
                resp = requests.get(
                    "https://ssd-api.jpl.nasa.gov/cad.api",
                    params={"date-min": f"{a}-01-01", "date-max": f"{b}-01-01",
                            "dist-max": dist_max, "diameter": "true"},
                    timeout=timeout)
                resp.raise_for_status()
                js = payload_json(resp, f"CAD {a}-{b}")
                n_declarados = int(js.get("count") or 0)
                if n_declarados:
                    campos_tramo, datos = js.get("fields"), js.get("data")
                    if not campos_tramo or not datos:
                        raise ErrorAPI(f"CAD {a}-{b}: count={n_declarados} pero la "
                                       f"respuesta no trae 'fields'/'data'")
                    if len(datos) != n_declarados:
                        # Sintoma tipico de respuesta cortada: se reintenta el tramo
                        # en vez de concatenar un trozo incompleto en silencio.
                        raise ErrorAPI(f"CAD {a}-{b}: count={n_declarados} pero llegaron "
                                       f"{len(datos)} filas (respuesta incompleta)")
                    if campos is None:
                        campos = campos_tramo
                    elif campos_tramo != campos:
                        # Si el orden de los campos cambiase entre tramos, concatenar
                        # las filas mezclaria columnas distintas sin error alguno.
                        raise ErrorAPI(f"CAD {a}-{b}: los campos difieren de los tramos "
                                       f"anteriores ({campos_tramo} != {campos})")
                    filas.extend(datos)
                else:
                    vacios.append(f"{a}-{b}")
                break
            except (requests.exceptions.RequestException, ErrorAPI) as exc:
                if intento == intentos - 1:
                    raise RuntimeError(f"CAD {a}-{b}: sin exito tras {intentos} "
                                       f"intentos ({exc})") from exc
                time.sleep(2 * (intento + 1))
        print(f"  {a}-{b}: {len(filas):,} eventos acumulados", end="\r")

    print(" " * 60, end="\r")
    if vacios:
        # Un tramo vacio es legitimo solo antes de los primeros descubrimientos;
        # a partir de ahi indica que la consulta devolvio menos de lo que deberia.
        print(f"⚠ Tramos sin eventos: {', '.join(vacios)}")
    if not filas:
        raise RuntimeError("La CAD API no devolvió ningún evento en todo el rango: "
                           "el dataset resultante seria inservible. Revisa dist_max, "
                           "el rango de fechas y la conectividad.")
    # Red de seguridad por si un evento cae justo en la frontera de dos tramos
    return pd.DataFrame(filas, columns=campos).drop_duplicates(subset=["des", "cd"])


def avisar_coercion(original, convertida, etiqueta):
    """Informa de los valores que `errors="coerce"` convirtio en NaN/NaT.

    La coercion silenciosa es deseable (las APIs mandan campos vacios), pero un
    cambio de formato en el origen tambien pasaria por aqui sin dejar rastro.
    """
    n = int((convertida.isna() & original.notna()).sum())
    if n:
        print(f"⚠ {etiqueta}: {n:,} valores no convertibles, quedan como nulos")
    return n


In [4]:
# Carga de datos

# Columnas que dan por hechas las celdas siguientes y notebooks/*.ipynb. Un CSV
# en cache generado por una version anterior del pipeline se cargaria sin error
# y reventaria mucho despues, en medio del analisis.
COLUMNAS_MINIMAS = ["Object", "Close-Approach (CA) Date", "CA DistanceNominal (au)",
                    "CA DistanceMinimum (au)", "V relative(km/s)", "V infinity(km/s)",
                    "H(mag)", "Diameter(km)"]

necesita_descarga = True
df = None

if os.path.exists(ruta_csv):
    print(f"✔ Archivo CSV encontrado en {ruta_csv}")
    if archivo_es_reciente(ruta_csv, dias_maximos=30):
        print("✔ Archivo actualizado (menos de 30 días). Cargando desde archivo local...")
        try:
            df_cache = pd.read_csv(ruta_csv)
        except (OSError, UnicodeDecodeError,
                pd.errors.ParserError, pd.errors.EmptyDataError) as e:
            print(f"✘ Cache ilegible ({type(e).__name__}: {e}).")
            print("  Se procederá a descargar datos frescos...")
        else:
            faltan = [c for c in COLUMNAS_MINIMAS if c not in df_cache.columns]
            if faltan:
                print(f"✘ Cache de una versión anterior del pipeline: faltan {faltan}.")
                print("  Se procederá a descargar datos frescos...")
            elif df_cache.empty:
                print("✘ Cache sin filas. Se procederá a descargar datos frescos...")
            else:
                df = df_cache
                df['Close-Approach (CA) Date'] = df['Close-Approach (CA) Date'].apply(limpiar_fecha)
                necesita_descarga = False
                print(f"✔ Datos cargados correctamente. {len(df):,} registros.")
    else:
        print("✘ Archivo desactualizado (más de 30 días). Se descargará de nuevo.")
else:
    print(f"✘ Archivo CSV no encontrado en {ruta_csv}")

 # DESCARGA DESDE LA API DE JPL

if necesita_descarga:
    print("\n⬇ Descargando datos desde la API de JPL (~6 min, 339k eventos)...")

    # dist-max: la CAD API lo fija en 0.05 au POR DEFECTO, que es exactamente el
    # umbral de distancia de la definicion PHA. Dejarlo implicito censura la
    # muestra en el umbral de la propia etiqueta: satura MOID<=0.05 (99.7% de los
    # objetos) y empobrece H<=22, de modo que PHA colapsa a un unico umbral.
    # 0.5 au es el limite superior que sirve la base de datos de JPL.
    DIST_MAX_AU = 0.5

    try:
        df = descargar_cad(dist_max=DIST_MAX_AU)
    except requests.exceptions.RequestException as e:
        print(f"✘ Error de conexión con la API de JPL: {e}")
        raise

    print(f"✔ Descarga completada. {len(df):,} registros recibidos.")
    print(f"  Columnas disponibles: {list(df.columns)}")

    # Campos sin los que el etiquetado posterior no tiene sentido: si la API
    # cambia de esquema, mejor parar aqui que construir un CSV incompleto.
    CAMPOS_API = ["des", "cd", "dist", "dist_min", "v_rel", "v_inf", "h"]
    faltan_api = [c for c in CAMPOS_API if c not in df.columns]
    if faltan_api:
        raise RuntimeError(f"La CAD API no devolvió los campos {faltan_api}; "
                           f"recibidos: {list(df.columns)}")

    # Selección de columnas (diameter/diameter_sigma solo si vienen)
    columnas_relevantes = CAMPOS_API + ["diameter", "diameter_sigma"]
    columnas_relevantes = [c for c in columnas_relevantes if c in df.columns]
    df = df[columnas_relevantes].copy()

    # Conversión numérica: se contabiliza lo que se pierde por el camino
    for col in ["dist", "dist_min", "v_rel", "v_inf", "h", "diameter", "diameter_sigma"]:
        if col in df.columns:
            convertida = pd.to_numeric(df[col], errors='coerce')
            avisar_coercion(df[col], convertida, f"campo '{col}' de la CAD API")
            df[col] = convertida

    # Diámetro estimado donde falta
    if "diameter" not in df.columns:
        df["diameter"] = float("nan")
    if "diameter_sigma" not in df.columns:
        df["diameter_sigma"] = float("nan")
    albedo = 1329 / math.sqrt(0.14)
    mascara_sin_diametro = df["diameter"].isna()
    df.loc[mascara_sin_diametro, "diameter"] = (
        albedo * (10 ** (-0.2 * df.loc[mascara_sin_diametro, "h"]))
    )
    df.loc[df["diameter_sigma"].isna(), "diameter_sigma"] = df["diameter"] * 0.35
    sin_diametro = int(df["diameter"].isna().sum())
    if sin_diametro:
        # Sin H no hay forma de imputar el diametro: queda NaN y el notebook de
        # ML descarta esas filas, asi que conviene saber cuantas son.
        print(f"⚠ {sin_diametro:,} eventos sin diámetro ni H para imputarlo (quedan NaN)")

    # Renombrar y guardar
    df = df.rename(columns={
        "des":            "Object",
        "cd":             "Close-Approach (CA) Date",
        "dist":           "CA DistanceNominal (au)",
        "dist_min":       "CA DistanceMinimum (au)",
        "v_rel":          "V relative(km/s)",
        "v_inf":          "V infinity(km/s)",
        "h":              "H(mag)",
        "diameter":       "Diameter(km)",
        "diameter_sigma": "Std Diameter(km)",
    })

    df.to_csv(ruta_csv, index=False)
    print(f"✔ CSV guardado en {ruta_csv}")

    df['Close-Approach (CA) Date'] = df['Close-Approach (CA) Date'].apply(limpiar_fecha)

if df is None:
    raise RuntimeError("No se pudieron cargar los datos: ni el cache local era "
                       "utilizable ni se intentó la descarga.")

print("\n           RESUMEN DE DATOS CARGADOS           ")
print(f"  Total de registros          : {len(df):,}")
print(f"  Nulos en H(mag)             : {df['H(mag)'].isna().sum():,}")
print(f"  Nulos en Diameter(km)       : {df['Diameter(km)'].isna().sum():,}")

print("\n            PRIMEROS 5 REGISTROS     ")
print(df[['Object', 'Diameter(km)', 'CA DistanceNominal (au)']].head().to_string(index=False))

if not necesita_descarga:
    print(f"\n Fuente: archivo local  →  {ruta_csv}")
    print(f"   Última modificación: {time.ctime(os.path.getmtime(ruta_csv))}")
else:
    print(f"\n Fuente: API de JPL (descarga fresca)")


✔ Archivo CSV encontrado en .\close_approaches.csv
✔ Archivo actualizado (menos de 30 días). Cargando desde archivo local...
✔ Datos cargados correctamente. 32,568 registros.

           RESUMEN DE DATOS CARGADOS           
  Total de registros          : 32,568
  Nulos en H(mag)             : 8
  Nulos en Diameter(km)       : 7

            PRIMEROS 5 REGISTROS     
    Object  Diameter(km)  CA DistanceNominal (au)
    509352      0.333013                 0.009632
2014 SC324      0.047039                 0.039964
2012 UK171      0.045547                 0.049706
  2024 BA5      0.022206                 0.026434
  2024 BW1      0.033609                 0.037979

 Fuente: archivo local  →  .\close_approaches.csv
   Última modificación: Thu Jun 18 20:00:07 2026


## Etiquetado de peligrosidad (PHA)

El objetivo del proyecto es **inferir el carácter potencialmente peligroso (PHA) de un NEO a partir únicamente de la cinemática de sus aproximaciones observadas**, sin usar los elementos orbitales que definen formalmente la etiqueta. Por eso construimos el *target* en dos versiones:

- **`PHA_official`** — el flag oficial `pha` de la [Small-Body Database (SBDB)](https://ssd-api.jpl.nasa.gov/doc/sbdb_query.html) de JPL. Es el *ground truth*. Un objeto es PHA si su **MOID ≤ 0.05 au** *y* su **magnitud absoluta H ≤ 22** (diámetro ≳ 140 m)[^1]. Traemos además `MOID` y `H` oficiales **solo para validar y etiquetar**, nunca como variables predictoras.
- **`PHA_proxy`** — etiqueta derivada *exclusivamente de los datos observados*: el objeto se marca peligroso si su **H observado ≤ 22** y la **mínima distancia de aproximación observada (`dist_min`) ≤ 0.05 au**, usando la distancia observada como aproximación del MOID. La comparación `proxy` vs `official` mide cuán bien la distancia observada sustituye al MOID orbital (sub-resultado del paper).

> **Caveat documentado:** el diámetro fue imputado desde H con la relación de albedo (`albedo = 1329/√0.14`)[^2] en la celda anterior cuando faltaba; por eso `H(mag)` y `Diameter(km)` no son independientes para esos registros. Las etiquetas se calculan a nivel de **objeto** (no de evento) y se almacenan denormalizadas en el mismo CSV para no alterar la estructura de un solo archivo del pipeline.

> **Metadatos de selección (no son features):** también traemos de la SBDB la **fecha real de primera observación** (`first_obs`), el **arco orbital** (`data_arc`) y el número de observaciones (`n_obs_used`). Permiten un análisis de la función de selección observacional riguroso (por fecha de descubrimiento, no por año del primer evento de aproximación) y medir el confundidor de caracterización orbital.

[^1]: Criterio oficial de Potentially Hazardous Asteroid. Fuente: [CNEOS FAQ](https://cneos.jpl.nasa.gov/faq/) (JPL/NASA, Center for Near Earth Object Studies).
[^2]: Relación estándar H–albedo–diámetro `D = 1329·10^(-0.2H)/√p_V`. Fuente: [CNEOS Asteroid Size Estimator](https://cneos.jpl.nasa.gov/tools/ast_size_est.html) (JPL/NASA), que cita Bowell et al. (1989), *Asteroids II*, pp. 524-556, y Harris & Harris (1997), *Icarus* 126:450-454.

In [5]:
ruta_sbdb = os.path.join(carpeta_destino, "sbdb_neo.csv")
sbdb_fields = ["pdes", "full_name", "pha", "neo", "moid", "H", "diameter",
               "first_obs", "last_obs", "data_arc", "n_obs_used"]

def _sbdb_cache_ok(ruta):
    """Cache valido solo si es reciente Y contiene todos los campos que pedimos."""
    if not archivo_es_reciente(ruta, dias_maximos=30):
        return False
    try:
        return set(sbdb_fields).issubset(pd.read_csv(ruta, nrows=1).columns)
    except (OSError, UnicodeDecodeError,
            pd.errors.ParserError, pd.errors.EmptyDataError) as e:
        # Se vuelve a descargar, pero dejando dicho por que: si no, un cache
        # corrupto se traduce en descargas repetidas sin explicacion.
        print(f"⚠ Cache SBDB inservible ({type(e).__name__}: {e}); se descargará de nuevo")
        return False

if _sbdb_cache_ok(ruta_sbdb):
    print(f"✔ Catalogo SBDB local reciente. Cargando {ruta_sbdb}...")
    sbdb = pd.read_csv(ruta_sbdb)
else:
    print("⬇ Descargando catalogo de NEOs desde la SBDB de JPL...")
    url_sbdb = "https://ssd-api.jpl.nasa.gov/sbdb_query.api"
    params_sbdb = {"fields": ",".join(sbdb_fields), "sb-group": "neo"}
    resp_sbdb = requests.get(url_sbdb, params=params_sbdb, timeout=120)
    resp_sbdb.raise_for_status()
    js = payload_json(resp_sbdb, "SBDB query")
    if not js.get("data") or not js.get("fields"):
        raise ErrorAPI(f"SBDB query: respuesta sin datos utilizables (claves: {list(js)})")
    sbdb = pd.DataFrame(js["data"], columns=js["fields"])
    faltan_sbdb = [c for c in sbdb_fields if c not in sbdb.columns]
    if faltan_sbdb:
        raise ErrorAPI(f"SBDB query: faltan los campos {faltan_sbdb} en la respuesta")
    sbdb.to_csv(ruta_sbdb, index=False)
    print(f"✔ Catalogo SBDB guardado. {len(sbdb):,} NEOs.")

# --- Normalizacion de tipos y union por designacion (Object == pdes) ---
sbdb["pdes"] = sbdb["pdes"].astype(str)
for _c in ["moid", "H", "data_arc", "n_obs_used"]:
    _conv = pd.to_numeric(sbdb[_c], errors="coerce")
    avisar_coercion(sbdb[_c], _conv, f"campo '{_c}' de la SBDB")
    sbdb[_c] = _conv
_foy = pd.to_datetime(sbdb["first_obs"], errors="coerce")
avisar_coercion(sbdb["first_obs"], _foy, "campo 'first_obs' de la SBDB")
sbdb["first_obs_year"] = _foy.dt.year
df["Object"] = df["Object"].astype(str)
mapa = sbdb.drop_duplicates("pdes").set_index("pdes")

# Columnas oficiales: etiqueta + validacion + metadatos de SELECCION.
# NINGUNA se usa como feature predictora (eso seria circular).
df["MOID (au)"]      = df["Object"].map(mapa["moid"])
df["H_SBDB(mag)"]    = df["Object"].map(mapa["H"])
df["first_obs_year"] = df["Object"].map(mapa["first_obs_year"])  # fecha REAL de 1a observacion
df["data_arc(d)"]    = df["Object"].map(mapa["data_arc"])        # arco orbital (caracterizacion)
df["n_obs_used"]     = df["Object"].map(mapa["n_obs_used"])      # nro de observaciones astrometricas
_pha_txt = df["Object"].map(mapa["pha"])
df["PHA_official"]   = _pha_txt.map({"Y": 1, "N": 0}).astype("Int8")
# La SBDB deja 'pha' vacio en objetos sin MOID calculado; cualquier OTRO valor
# significaria que cambio la codificacion, y el mapeo lo convertiria en <NA>
# haciendo desaparecer esos objetos del analisis sin ningun aviso.
_pha_raros = sorted(set(_pha_txt[_pha_txt.notna() & df["PHA_official"].isna()].unique()))
if _pha_raros:
    raise ValueError(f"La columna 'pha' de la SBDB trae valores no reconocidos "
                     f"{_pha_raros[:5]}: se esperaba solo 'Y'/'N'/vacio")

# --- Evento OBSERVADO vs calculado retroactivamente ---
# La CAD API integra hacia atras hasta 1900 tambien para objetos descubiertos
# despues: esos eventos son salidas de un modelo dinamico, no observaciones.
# Se marcan para poder restringir el analisis principal a lo realmente
# observado y dejar el catalogo completo para el anexo de sensibilidad.
_ca_dt = pd.to_datetime(df["Close-Approach (CA) Date"].apply(limpiar_fecha),
                        format="%Y-%b-%d %H:%M", errors="coerce")
# Una fecha que no encaje en el formato se convierte en NaT, y NaT >= año es
# False: el evento quedaria marcado como PRE-descubrimiento y desapareceria del
# analisis principal como si fuese una integracion retroactiva. Se marca como
# desconocido (<NA>) y se aborta si el formato ha cambiado para muchas filas.
_n_fechas_malas = int(_ca_dt.isna().sum())
if _n_fechas_malas:
    _frac_malas = _n_fechas_malas / len(df)
    print(f"⚠ {_n_fechas_malas:,} fechas de aproximación no parseables "
          f"({100*_frac_malas:.2f}%): post_discovery queda indefinido en ellas")
    if _frac_malas > 0.01:
        raise ValueError(f"{100*_frac_malas:.1f}% de las fechas no siguen el formato "
                         f"'%Y-%b-%d %H:%M': la CAD API cambió de formato y el filtro "
                         f"post-descubrimiento no es fiable")
_ca_year = _ca_dt.dt.year
df["post_discovery"] = ((_ca_year >= df["first_obs_year"])
                        .mask(df["first_obs_year"].isna() | _ca_year.isna())
                        .astype("Int8"))

# --- Etiqueta PROXY: propiedades por OBJETO desde lo realmente OBSERVADO ---
# Solo eventos post-descubrimiento: incluir los integrados inflaria la calidad
# aparente del proxy (corr con el MOID sube de 0.868 a 0.931 si se incluyen).
_obs = df[df["post_discovery"] == 1]
distmin_obj = df["Object"].map(_obs.groupby("Object")["CA DistanceMinimum (au)"].min())
H_obj       = df["Object"].map(_obs.groupby("Object")["H(mag)"].min())
df["PHA_proxy"] = ((H_obj <= 22) & (distmin_obj <= 0.05)).astype("Int8")

# --- Guardar CSV enriquecido (un unico archivo, estructura intacta) ---
df.to_csv(ruta_csv, index=False)
print(f"✔ CSV actualizado con etiquetas en {ruta_csv}")

# --- Snapshot CONGELADO y fechado para reproducibilidad del paper ---
# Se congela UNA sola vez: si ya existe un close_approaches_v*.csv no se crea
# otro ni se toca snapshot_info.json (protege el snapshot citado en el paper).
# Para re-congelar con datos nuevos: borrar los close_approaches_v*.csv.
import json as _json, sys as _sys, glob as _glob
_snaps = sorted(_glob.glob(os.path.join(carpeta_destino, "close_approaches_v*.csv")))
if _snaps:
    print(f"✔ Snapshot ya congelado: {_snaps[-1]}  (no se sobrescribe; ver snapshot_info.json)")
else:
    fecha_snap = datetime.utcnow().strftime("%Y%m%d")
    ruta_snap = os.path.join(carpeta_destino, f"close_approaches_v{fecha_snap}.csv")
    df.to_csv(ruta_snap, index=False)
    _info = {"snapshot_date_utc": fecha_snap,
             # dist_max_au queda registrado: es el parametro cuyo valor por
             # defecto (0.05) censuraba la muestra en el umbral de la etiqueta.
             "dist_max_au": 0.5,
             "n_events": int(len(df)),
             "n_events_post_discovery": int((df["post_discovery"] == 1).sum()),
             "n_objects": int(df["Object"].nunique()),
             "n_pha_official": int((df.drop_duplicates("Object")["PHA_official"] == 1).sum()),
             "python": _sys.version.split()[0], "pandas": pd.__version__}
    with open(os.path.join(carpeta_destino, "snapshot_info.json"), "w",
              encoding="utf-8") as _fj:
        _json.dump(_info, _fj, indent=2)
    print(f"✔ Snapshot congelado: {ruta_snap}  (ver snapshot_info.json)")

# --- Resumen / validacion fisica (a nivel OBJETO) ---
obj = df.drop_duplicates("Object")
n_match = int(obj["PHA_official"].notna().sum())
print("\n        RESUMEN DE ETIQUETADO (por objeto)        ")
print(f"  Objetos unicos              : {len(obj):,}")
print(f"  Emparejados con SBDB        : {n_match:,} ({100*n_match/len(obj):.1f}%)")
print(f"  PHA_official = 1            : {int((obj['PHA_official']==1).sum()):,}")
print(f"  PHA_proxy    = 1            : {int((obj['PHA_proxy']==1).sum()):,}")

n_obs_ev = int((df["post_discovery"] == 1).sum())
print(f"  Eventos observados          : {n_obs_ev:,} de {len(df):,} "
      f"({100*n_obs_ev/len(df):.1f}%; el resto son integrados hacia atras)")

if n_match == 0:
    raise RuntimeError("Ningún objeto del catálogo CAD emparejó con la SBDB: la "
                       "designacion 'des' y 'pdes' ya no son comparables, no hay "
                       "etiquetas que analizar.")

val = obj.dropna(subset=["PHA_official"])
tp = int(((val["PHA_proxy"]==1) & (val["PHA_official"]==1)).sum())
fp = int(((val["PHA_proxy"]==1) & (val["PHA_official"]==0)).sum())
fn = int(((val["PHA_proxy"]==0) & (val["PHA_official"]==1)).sum())
prec = tp/(tp+fp) if tp+fp else float("nan")
rec  = tp/(tp+fn) if tp+fn else float("nan")
print(f"  Proxy vs oficial           : precision={prec:.3f}  recall={rec:.3f}")
# Correlacion proxy-MOID por objeto, solo sobre eventos observados
_dmin_obs = _obs.groupby("Object")["CA DistanceMinimum (au)"].min()
_vm = pd.DataFrame({"dmin": _dmin_obs,
                    "moid": mapa["moid"].reindex(_dmin_obs.index)}).dropna()
print(f"  corr(dist_min observ., MOID): {_vm['dmin'].corr(_vm['moid']):.3f}")

✔ Catalogo SBDB local reciente. Cargando .\sbdb_neo.csv...


✔ CSV actualizado con etiquetas en .\close_approaches.csv
✔ Snapshot ya congelado: .\close_approaches_v20260619.csv  (no se sobrescribe; ver snapshot_info.json)

        RESUMEN DE ETIQUETADO (por objeto)        
  Objetos unicos              : 18,934
  Emparejados con SBDB        : 18,927 (100.0%)
  PHA_official = 1            : 1,379
  PHA_proxy    = 1            : 1,401
  Proxy vs oficial           : precision=0.968  recall=0.983
  corr(dist_min observ., MOID): 0.725


---

## Datos listos

El dataset procesado ha sido guardado como `close_approaches.csv` en esta misma carpeta (`data/`).

Para continuar con el analisis estadistico y machine learning, ejecutar el notebook `ProyectoNeoRework_ml.ipynb`.